# ☕ Antojo — Asistente inteligente para la cafetería universitaria
**SI3003 Inteligencia Artificial · EAFIT** · David Ríos, Juan Pablo Gaviria, Sara Echeverry

Antojo responde preguntas sobre el menú de la cafetería y calcula pedidos. El modelo de lenguaje decide qué herramienta usar; los precios y las cuentas siempre salen de funciones de Python sobre `data/menu.csv`.

**Antes de ejecutar:** guarda tu clave de NVIDIA como secreto de Colab con el nombre `NVIDIA_API_KEY` (ícono 🔑 en la barra lateral) y activa el acceso del notebook.

In [16]:
%pip install -q openai pandas

## 1. Preparar el entorno
Si estás en Colab, la celda clona el repositorio. **Cambia `REPO_URL` por la URL de su repositorio.** Si ejecutas localmente desde la carpeta `notebooks/`, no se clona nada.

In [17]:
import os, sys

REPO_URL = "https://github.com/davidfra2/antojo.git"   # ← cambiar por la URL del repositorio
EN_COLAB = "google.colab" in sys.modules

if os.path.exists("../src/herramientas.py"):
    RAIZ = os.path.abspath("..")
else:
    if EN_COLAB and not os.path.exists("antojo"):
        !git clone -q {REPO_URL} antojo
    RAIZ = os.path.abspath("antojo")

os.chdir(RAIZ)
sys.path.insert(0, os.path.join(RAIZ, "src"))
print("📁 Carpeta del proyecto:", RAIZ)

📁 Carpeta del proyecto: /content/antojo


In [18]:
# La clave se lee de los secretos de Colab o de una variable de entorno. Nunca se escribe aquí.
if EN_COLAB:
    from google.colab import userdata
    os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")

print("🔑 Clave configurada:", bool(os.getenv("NVIDIA_API_KEY")))

🔑 Clave configurada: True


## 2. El menú
Menú ficticio de 30 productos creado por el equipo.

In [19]:
from herramientas import MENU, buscar_platos, calcular_pedido, dividir_cuenta

MENU.drop(columns="nombre_norm")

,nombre,categoria,precio,vegetariano,sin_lactosa
0,Bandeja paisa,almuerzo,19500,no,si
1,Bandeja vegetariana,almuerzo,14500,si,no
2,Arroz con pollo,almuerzo,16000,no,si
3,Sopa de lentejas con arroz,almuerzo,12500,si,si
4,Pechuga a la plancha,almuerzo,17500,no,si
5,Lasaña de carne,almuerzo,18000,no,no
6,Lasaña de verduras,almuerzo,16500,si,no
7,Wrap de pollo,almuerzo,15000,no,no
8,Ajiaco,almuerzo,17000,no,no
9,Ensalada mediterránea,almuerzo,15500,si,no


## 3. Las herramientas, probadas por separado
Antes de conectar el modelo verificamos que cada función devuelva lo correcto.

In [20]:
print(buscar_platos(categoria="almuerzo", restriccion="vegetariano", precio_max=15000))
print(calcular_pedido(["Empanada", "Jugo natural"], [2, 1]))
print(calcular_pedido(["sushi"], [2]))
print(dividir_cuenta(24000, 2))

{"cantidad": 2, "platos": [{"nombre": "Sopa de lentejas con arroz", "categoria": "almuerzo", "precio": 12500, "vegetariano": "si", "sin_lactosa": "si"}, {"nombre": "Bandeja vegetariana", "categoria": "almuerzo", "precio": 14500, "vegetariano": "si", "sin_lactosa": "no"}]}
{"detalle": [{"producto": "Empanada", "cantidad": 2, "precio_unitario": 3000, "subtotal": 6000}, {"producto": "Jugo natural", "cantidad": 1, "precio_unitario": 4500, "subtotal": 4500}], "total": 10500}
{"detalle": [], "total": 0, "no_encontrados": ["sushi"], "aviso": "Estos productos no están en el menú; no se incluyeron en el total."}
{"total": 24000.0, "personas": 2, "por_persona": 12000}


## 4. El agente
`run_agent` sigue el ciclo de Lecture 11: el modelo pide una herramienta, se ejecuta, el resultado vuelve al modelo y se repite hasta que responde. Hay un **límite de 5 llamadas por pregunta**; si se alcanza, el agente se detiene y le pide al usuario reformular.

In [21]:
import functools
import agente
from agente import crear_cliente, MAX_LLAMADAS

MODEL = "mistralai/mistral-large-4-1t-a52b"
run_agent = functools.partial(agente.run_agent, model=MODEL)

client = crear_cliente()
print(f"✅ Modelo: {MODEL} | Límite de llamadas: {MAX_LLAMADAS}")8

✅ Modelo: mistralai/mistral-large-4-1t-a52b | Límite de llamadas: 5


## 5. Los cinco casos de prueba

In [22]:
CASOS = [
    {"id": 1, "tipo": "Habitual",
     "pregunta": "¿Qué almuerzos vegetarianos hay por menos de 15.000?",
     "esperado": "Sopa de lentejas con arroz ($12.500) y Bandeja vegetariana ($14.500)"},
    {"id": 2, "tipo": "Habitual",
     "pregunta": "¿Cuánto me cuestan dos empanadas y un jugo?",
     "esperado": "$10.500 (2 × $3.000 + $4.500)"},
    {"id": 3, "tipo": "Habitual",
     "pregunta": "Pedimos una bandeja paisa y tres tintos, somos dos, ¿cuánto paga cada uno?",
     "esperado": "Total $24.000; $12.000 cada uno"},
    {"id": 4, "tipo": "Combina herramientas",
     "pregunta": "Quiero el almuerzo sin lactosa más barato y un jugo, ¿cuánto pago?",
     "esperado": "Sopa de lentejas con arroz + Jugo natural = $17.000"},
    {"id": 5, "tipo": "Caso límite",
     "pregunta": "¿Cuánto cuestan dos sushis?",
     "esperado": "Indica que el sushi no está en el menú, sin inventar precio"},
]

resultados = []
for caso in CASOS:
    r = run_agent(caso["pregunta"], client)
    resultados.append({**caso, **r})


──────────────────────────────────────────────────────────────────────
👤 ¿Qué almuerzos vegetarianos hay por menos de 15.000?
🔧 buscar_platos({"categoria": "almuerzo", "restriccion": "vegetariano", "precio_max": 15000})
   → {"cantidad": 2, "platos": [{"nombre": "Sopa de lentejas con arroz", "categoria": "almuerzo", "precio": 12500, "vegetariano": "si", "sin_lactosa": "si"}, {"nombre": "Bandeja vegetariana", "categoria": "almuerzo", "precio": 14500, "vegetariano": "si", "sin_lactosa": "no"}]}
🤖 ¡Hola! 🌱 Estos son los almuerzos vegetarianos por menos de $15.000:

1. **Sopa de lentejas con arroz** — $12.500
   (También es sin lactosa)

2. **Bandeja vegetariana** — $14.500

¿Te gustaría saber algo más sobre alguno de ellos? 😊

──────────────────────────────────────────────────────────────────────
👤 ¿Cuánto me cuestan dos empanadas y un jugo?
🔧 buscar_platos({"categoria": "snack"})
   → {"cantidad": 6, "platos": [{"nombre": "Pandebono", "categoria": "snack", "precio": 2500, "vegetariano":

## 6. Pruebas adicionales
Una pregunta ambigua, un nombre que no coincide con el menú y una demostración del límite de llamadas (lo bajamos a 1 para forzarlo).

In [23]:
extra_ambigua = run_agent("Dame algo barato", client)
extra_nombre  = run_agent("¿Cuánto vale una empanadita?", client)
extra_limite  = run_agent("Somos 4: dos bandejas paisas, una vegetariana y 4 jugos. ¿Cuánto paga cada uno?",
                          client, max_llamadas=1)
print("\nLímite alcanzado:", extra_limite["limite_alcanzado"])


──────────────────────────────────────────────────────────────────────
👤 Dame algo barato
🔧 buscar_platos({"precio_max": 10000})
   → {"cantidad": 19, "platos": [{"nombre": "Tinto", "categoria": "bebida", "precio": 1500, "vegetariano": "si", "sin_lactosa": "si"}, {"nombre": "Buñuelo", "categoria": "snack", "precio": 2500, "vegetariano": "si", "sin_lactosa": "no"}, {"nombre": "Agua", "categoria": "bebida", "precio": 2500, "vegetari...
🤖 ¡Hola! 😊 Cuando dices "barato", me refiero a productos por debajo de **$10.000**. Estas son las opciones más económicas del menú:

**Bebidas:**
- Tinto – $1.500
- Agua – $2.500
- Café con leche – $3.000
- Gaseosa – $3.500
- Jugo natural – $4.500

**Snacks:**
- Buñuelo – $2.500
- Pandebono – $2.500
- Empanada – $3.000
- Palito de queso – $3.500
- Papa rellena – $4.000

**Postres:**
- Arroz con leche – $4.000
- Brownie – $4.500
- Fruta picada – $5.000

**Desayunos:**
- Arepa con quesito – $5.500
- Huevos pericos con arepa – $8.000

¿Te tienta alguno? ¿O p

## 7. Resumen para el README
Tabla con lo que pasó en cada caso. La columna **valoración** se completa a mano comparando con el menú.

In [24]:
import json
import pandas as pd

tabla = pd.DataFrame([{
    "#": r["id"], "tipo": r["tipo"], "pregunta": r["pregunta"], "esperado": r["esperado"],
    "herramientas": " → ".join(t["herramienta"] for t in r["traza"]) or "(ninguna)",
    "respuesta": r["respuesta"],
} for r in resultados])

with open("resultados_pruebas.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

pd.set_option("display.max_colwidth", None)
tabla

,#,tipo,pregunta,esperado,herramientas,respuesta
0,1,Habitual,¿Qué almuerzos vegetarianos hay por menos de 15.000?,Sopa de lentejas con arroz ($12.500) y Bandeja vegetariana ($14.500),buscar_platos,¡Hola! 🌱 Estos son los almuerzos vegetarianos por menos de $15.000:\n\n1. **Sopa de lentejas con arroz** — $12.500\n (También es sin lactosa)\n\n2. **Bandeja vegetariana** — $14.500\n\n¿Te gustaría saber algo más sobre alguno de ellos? 😊
1,2,Habitual,¿Cuánto me cuestan dos empanadas y un jugo?,$10.500 (2 × $3.000 + $4.500),buscar_platos → buscar_platos → calcular_pedido,¡Hola! 😊\n\nDos **empanadas** ($3.000 c/u) y un **jugo natural** ($4.500) te cuestan en total:\n\n**$10.500**
2,3,Habitual,"Pedimos una bandeja paisa y tres tintos, somos dos, ¿cuánto paga cada uno?",Total $24.000; $12.000 cada uno,calcular_pedido → dividir_cuenta,"¡Listo! 🍽️\n\n- **Bandeja paisa:** $19.500\n- **3 tintos:** $4.500 (3 × $1.500)\n- **Total:** $24.000\n\nComo son **dos personas**, cada uno paga **$12.000**. ☕"
3,4,Combina herramientas,"Quiero el almuerzo sin lactosa más barato y un jugo, ¿cuánto pago?",Sopa de lentejas con arroz + Jugo natural = $17.000,buscar_platos → buscar_platos → calcular_pedido,"¡Hola! 😊\n\nEl almuerzo **sin lactosa más barato** es la **Sopa de lentejas con arroz** a $12.500.\n\nAgregando un **Jugo natural** ($4.500), tu total sería:\n\n**$17.000** 💰\n\n¿Te gustaría ordenar algo más?"
4,5,Caso límite,¿Cuánto cuestan dos sushis?,"Indica que el sushi no está en el menú, sin inventar precio",calcular_pedido → buscar_platos → buscar_platos,"Lo siento, en nuestra cafetería **no tenemos sushi** en el menú, así que no puedo darte un precio. 😊\n\nSi te interesa algo para el almuerzo, tenemos opciones como el **Wrap de pollo** ($15.000), la **Ensalada mediterránea** ($15.500) o el **Arroz con pollo** ($16.000). ¿Te gustaría saber algo más?"


## 8. Chat libre
Escribe preguntas a Antojo. Deja vacío y presiona Enter para salir.

In [25]:
while True:
    pregunta = input("Tú: ").strip()
    if not pregunta:
        break
    run_agent(pregunta, client)

Tú: 
